# Tutorial: CoF condensate bursting model

This notebook is a guide to **running** the model and **understanding the variables**
you'd change to explore it or reproduce the paper's sweeps. The equations, biological
motivation, and derivations behind each rate law are in the paper's Methods section —
this notebook does not re-derive them.

Run each cell in order. By the end you'll have run a single simulated cell, compared
the two rate laws, added cell-to-cell heterogeneity, and run the same `run_sweep`
function used to produce the paper's figures — just at a smaller scale, all within
this notebook.

## Part 0: Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from transcription_models import (
    ThreeStateModel, gillespie_simulation, run_sweep,
    calculate_rates_binding, calculate_rates_condensate,
)
from transcription_models.analysis import analyze_bursts
from transcription_models.heterogeneity import jitter_concentrations, sample_rate_parameters

In [ ]:
# Plot style and the two colors used everywhere below: gray for the binding
# (soluble) model, green for the condensate model — same convention as the paper figures.
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
COLOR_BINDING = '#555555'
COLOR_CONDENSATE = '#2d8a4e'

## Part 1: A single simulated cell

The three-state promoter (OFF → paused → ON) and the Gillespie algorithm that simulates
it are described in Methods. Here we just configure `ThreeStateModel` and run it.

### Step 1: Choose the promoter topology and rate constants

`ThreeStateModel` has states **1** (OFF), **2** (paused, TF bound), **3** (ON,
transcribing). The variables you'll
touch most:

| Variable | Meaning |
|---|---|
| `k2` | rate of 1 → 2, TF binding |
| `koff_2` | rate of 2 → 1, TF unbinding |
| `k3` | rate of 2 → 3 — this is the transition CoF acts on |
| `koff_3_to_2` | rate of 3 → 2 |
| `kt` | transcription rate while in state 3 |
| `allow_3_to_1` / `allow_3_to_2` | which reverse transition(s) out of state 3 exist |

The paper always uses `allow_3_to_1=False, allow_3_to_2=True`

In [ ]:
model = ThreeStateModel(
    k2=0.05, koff_2=0.5, k3=0.15, koff_3_to_2=0.5, kt=1.5,
    allow_3_to_1=False, allow_3_to_2=True,
)

### Step 2: Run the Gillespie simulation

`gillespie_simulation` takes the `model` plus two variables:

- `t_max` — how long to simulate, in the same time units as the rate constants above.
- `seed` — fixes the random draws so the trajectory is reproducible; change it (or drop
  it) to see a different stochastic realization of the same model.

In [ ]:
result = gillespie_simulation(model, t_max=500, seed=42)

`result` is a `SimulationResult` with three arrays you'll use throughout this notebook:
`.times` (event times), `.states` (1/2/3 at each event), and `.transcript_counts`
(cumulative transcripts at each event).

In [ ]:
print(f'{len(result.times)} events over t=[0, {result.times[-1]:.1f}]')
print(f'states visited: {sorted(s + 1 for s in set(result.states))}')
print(f'transcripts by the end: {result.transcript_counts[-1]}')

### Step 3: Plot the trajectory and summarize its bursts

`analyze_bursts` turns the raw trajectory into the summary statistics used in the
figures: `n_bursts`, `mean_burst_duration`, `mean_burst_size`, `burst_frequency`, and
`fraction_time_active`.

A **burst** here is a continuous period spent in the active state (state 3, i.e. the
gene is ON and transcribing) — it starts when the promoter enters state 3 and ends
when it leaves. `burst_frequency` (inverse unit time) is what Figure 4's panels
plot as **"Activation frequency"**; `mean_burst_size` (average transcripts
produced per burst) is what Figure 5's panels plot as **"Mean transcripts per active
state"**. Both are computed directly from this trajectory's bursts, the same way
they're computed per-cell across the full simulated population in `generate_figures.ipynb`.

In [ ]:
stats = analyze_bursts(result)
stats

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axes[0].step(result.times, result.states + 1, where='post')
axes[0].set_ylabel('State')
axes[0].set_yticks([1, 2, 3])
axes[1].step(result.times, result.transcript_counts, where='post', color=COLOR_CONDENSATE)
axes[1].set_ylabel('Transcripts')
axes[1].set_xlabel('Time')
plt.suptitle(f"Bursts: {stats['n_bursts']}, total transcripts: {result.transcript_counts[-1]}")
plt.tight_layout()

## Part 2: Binding vs. condensate rate laws

CoF concentration (`CoF_conc`) sets `k3` — the rate constant highlighted in Part 1 —
in two different ways, implemented as two functions with a shared call signature. The
equations behind each are in Methods; here we look at their variables and what they
return.

### Step 1: Shared kinetic parameters

Both `calculate_rates_binding` and `calculate_rates_condensate` take a `TF_conc` and a
`CoF_conc`, plus nominal rate constants, and return `(k2, k3, kt)`. The paper fixes
`TF_conc=1.0` (`tf_fixed`, see the config dict built in Part 4) and sweeps `CoF_conc`.

`calculate_rates_condensate` additionally takes:

| Variable | Meaning |
|---|---|
| `CoF_threshold` | CoF concentration at which condensation kicks in (CoF\*) |
| `k3_low` / `k3_max` | `k3` below / at-or-above threshold |
| `PR_max` | partition ratio in the condensed phase |
| `dense_phase_CoF` | effective CoF concentration inside the condensate |

In [ ]:
TF_conc = 1.0
CoF_threshold = 1.0
CoF_grid_fine = np.logspace(-1, 1, 100)

### Step 2: Evaluate both rate laws across a CoF sweep

`[1]` below pulls out `k3` (the second value of the returned `(k2, k3, kt)` tuple) at
each CoF value, so we can see the two rate laws side by side.

In [ ]:
k2_binding = [
    calculate_rates_binding(
        TF_conc, c, k2_base=0.02, k3_base=0.1, koff_2=0.5, koff_3=0.5, kt_base=1.0,
    )[1]
    for c in CoF_grid_fine
]
k2_condensate = [
    calculate_rates_condensate(
        TF_conc, c, k2_base=0.02, k3_low=0.1, k3_max=1.0,
        CoF_threshold=CoF_threshold, PR_max=1.0,
        koff_2=0.5, koff_3=0.5, kt_base=1.0, dense_phase_CoF=10.0,
    )[1]
    for c in CoF_grid_fine
]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(CoF_grid_fine, k2_binding, label='Binding (soluble)', color=COLOR_BINDING, lw=2)
ax.plot(CoF_grid_fine, k2_condensate, label='Condensate', color=COLOR_CONDENSATE, lw=2)
ax.axvline(CoF_threshold, color='gray', ls='--', lw=1)
ax.set_xscale('log')
ax.set_xlabel(r'$[\mathrm{CoF}]_\mathrm{nucleus}$')
ax.set_ylabel('Effective k3 rate')
ax.legend()
plt.tight_layout()

## Part 3: Cell-to-cell heterogeneity

The paper's simulated populations add two independent, per-cell sources of noise on
top of the nominal model: each cell's true CoF concentration is jittered around the
nominal condition (`jitter_concentrations`), and each cell draws its own rate
constants around the population means (`sample_rate_parameters`).

### Step 1: Jittered CoF per cell

`jitter_concentrations(conc_value, n_cells, jitter_CV, seed)` draws `n_cells`
log-normal samples around `conc_value`. `jitter_CV` is the coefficient of variation
(std / mean) of that spread — the paper uses `jitter_CV=0.12`. This jittered value
drives **both** the simulation physics and the x-axis position in scatter plots like
`Figures/Output/*/2C_black.png`.

In [ ]:
nominal_CoF = 1.0
jittered_CoF = jitter_concentrations(nominal_CoF, n_cells=500, jitter_CV=0.12, seed=42)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(jittered_CoF, bins=30, color=COLOR_CONDENSATE, alpha=0.7)
ax.axvline(nominal_CoF, color='k', ls='--', label='nominal')
ax.set_xlabel('Jittered CoF')
ax.set_ylabel('Cell count')
ax.legend()
plt.title('Per-cell CoF variability (CV=0.12)')
plt.tight_layout()

### Step 2: Per-cell rate parameters

`sample_rate_parameters` takes the same nominal rate constants used in Part 2, plus a
`*_CV` for each one, and returns a per-cell dict of sampled values. The paper sets
on-rate CVs (`k2_CV`, `k3_low_CV`, `k3_max_CV`, `kt_CV`) to `0.4` and off-rate CVs
(`koff_2_CV`, `koff_3_CV`) to `0.0` — the same values used in the full run (see the
config dict built in Part 4).

In [ ]:
cell_rates = sample_rate_parameters(
    k2_base=0.02, k3_low=0.1, k3_max=1.0, kt_base=1.0, PR_max=1.0,
    koff_2=0.5, koff_3=0.5,
    k2_CV=0.4, k3_low_CV=0.4, k3_max_CV=0.4, kt_CV=0.4,
    koff_2_CV=0.0, koff_3_CV=0.0, seed=0,
)
cell_rates

## Part 4: Mini dose-response sweep

Parts 1–3 are exactly what `run_sweep` (`transcription_models/sweep.py`) does
internally, for a population of cells rather than one: for each nominal CoF value, it
simulates `n_trajectories` cells, each with jittered CoF and per-cell rates, and
records every cell's burst statistics. This is the same function used to produce the
paper's figures, at full scale — here we call it directly, with a much smaller grid.

### Step 1: Build a config dict

`run_sweep` takes a `cfg` dict shaped like the one built below. The paper's full run
uses `cof_grid.n_points=20` and `simulation.n_trajectories=500`; here we use 5 and 30.

In [ ]:
mini_cfg = {
    'tf_fixed': TF_conc,
    'cof_grid': {'log10_min': -0.5, 'log10_max': 0.5, 'n_points': 5},
    'dense_phase_cof': 10.0,
    'off_gene_dilute_conc': 1.0,
    'rates': {
        'k2_base_binding': 0.02, 'k3_base_binding': 0.1,
        'koff_2': 0.5, 'koff_3': 0.5, 'kt_base': 1.0,
        'kt_cof_factor': 0.0, 'pr_max': 1.0,
    },
    'simulation': {'n_trajectories': 30, 't_max': 500, 'window_size': 500, 'base_seed': 0},
    'heterogeneity': {
        'cof_threshold_cv': 0.0, 'jitter_cv': 0.12,
        'k2_cv': 0.4, 'k3_low_cv': 0.4, 'k3_max_cv': 0.4, 'kt_cv': 0.4, 'pr_cv': 0.0,
        'koff_2_cv': 0.4, 'koff_3_cv': 0.4, 'p_gene_localization': 1.0,
    },
}

### Step 2: Run the sweep

`CoF_threshold_nom` is the nominal condensation threshold for this sweep (only used
when `model_type='condensate'`).

In [ ]:
sweep_result = run_sweep(mini_cfg, model_type='condensate', CoF_threshold_nom=1.0)

`sweep_result` holds one entry per per-cell metric (`transcripts`, `burst_freq`,
`burst_size`, `x_jittered`, ...), each shaped `{0: {grid_index: [per-cell values]}}` —
the outer `0` is a bookkeeping artifact from windowed sweeps and is always `0` here.

In [ ]:
n_conditions = mini_cfg['cof_grid']['n_points']
xs = np.concatenate([sweep_result['x_jittered'][0][j] for j in range(n_conditions)])
ys = np.concatenate([sweep_result['transcripts'][0][j] for j in range(n_conditions)])
print(f'{len(xs)} cells across {n_conditions} nominal CoF conditions')

### Step 3: Plot the dose-response scatter

Each point is one simulated cell: its jittered CoF (x) against its final transcript
count (y). Compare against `Figures/Output/*/2C_black.png` — same idea, full 20×500
sweep.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(xs, ys, s=12, alpha=0.5, color=COLOR_CONDENSATE)
ax.axvline(CoF_threshold, color='gray', ls='--')
ax.set_xscale('log')
ax.set_xlabel(r'$[\mathrm{CoF}]_\mathrm{nucleus}$')
ax.set_ylabel('Transcript count')
plt.title(f'Mini condensate sweep ({n_conditions}\u00d7{mini_cfg["simulation"]["n_trajectories"]} cells)')
plt.tight_layout()